In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, widgets
from IPython.display import display

import Whisker_Curation.Heatmap.whisker_heatmap as wh

In [ ]:
# Configuration
CSV_PATH = "C:\\Users\\wanglab\\Desktop\\Mel\\shortened\\lines_output.csv"
FRAMES_PER_GROUP = 100
HEATMAP_BINS = 100
COLORMAP = 'hot'
SHOW_WHISKER_OVERLAY = True

In [ ]:
# Load data
df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df)} rows")
print(f"Total frames: {df['Frame'].nunique()}")

# Get frame groups
groups = wh.get_frame_groups(df, num_frames=FRAMES_PER_GROUP)
print(f"Total frame groups: {len(groups)}")
print(f"\nFrame groups:")
for i, (start, end) in enumerate(groups[:10]):  # Show first 10
    print(f"  Group {i}: Frames {start}-{end}")
if len(groups) > 10:
    print(f"  ... and {len(groups)-10} more groups")

In [ ]:
# Interactive widget to scroll through frame groups
def update_heatmap(group_index):
    plt.close('all')  # Close previous plots
    
    start_frame, end_frame = groups[group_index]
    
    fig, ax = wh.plot_whisker_heatmap(
        df, 
        start_frame, 
        num_frames=FRAMES_PER_GROUP,
        bins=HEATMAP_BINS,
        cmap=COLORMAP,
        show_whiskers=SHOW_WHISKER_OVERLAY
    )
    
    plt.show()

# Create slider widget
slider = widgets.IntSlider(
    value=0,
    min=0,
    max=len(groups)-1,
    step=1,
    description='Frame Group:',
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='600px')
)

# Display info about current group
info_label = widgets.Label(value=f"Group 0: Frames {groups[0][0]}-{groups[0][1]}")

def on_slider_change(change):
    idx = change['new']
    start, end = groups[idx]
    info_label.value = f"Group {idx}: Frames {start}-{end}"

slider.observe(on_slider_change, names='value')

# Display widgets
display(widgets.VBox([info_label, slider]))

# Create interactive output
interactive_output = widgets.interactive_output(update_heatmap, {'group_index': slider})
display(interactive_output)